# NB_DL_RAW — Deep Learning sobre sinal bruto EEG (chb06)

Usa diretamente os `.npz` do **NB1** (`data/signals/`) — sinal EEG filtrado e reamostrado,
sem extração de features. Compara duas arquiteturas adaptadas para sinal temporal:

| Modelo | Entrada | Descrição |
|--------|---------|-----------|
| **EEGNet** | `(batch, 1, n_ch, T)` | Arquitetura específica EEG, ~2.5K params, projetada para n pequeno |
| **CNN-1D temporal** | `(batch, n_ch, T)` | Conv sobre o eixo do tempo, aprende filtros espectrais |

## Por que sinal bruto?
Com features manuais (NB3), DL não tem vantagem sobre RF — os dados já estão
pré-processados. Com sinal bruto, CNN/EEGNet podem aprender filtros temporais
diretamente e capturar padrões que as features manuais perdem.

## Protocolo
- Mesmo LOSO do NB4 v5 (seleção de janela por inner loop)
- Janelas de **4s** com passo **2s** sobre o sinal (menor que NB3 para caber na VRAM)
- Nível R4 fixo (15 canais)
- 1 seed para rodar rápido — mude para 5 para resultado final
- GTX 1650 4GB: cada batch ~7 MB — cabe confortavelmente


In [1]:
# ── Instalação ───────────────────────────────────────────────────────────────
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
%pip install -q numpy pandas tqdm scikit-learn
print('Concluído. Reinicie o kernel se PyTorch foi instalado agora.')


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Concluído. Reinicie o kernel se PyTorch foi instalado agora.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ── Imports e GPU ────────────────────────────────────────────────────────────
import json, warnings, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU:  {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('  AVISO: CUDA não disponível — CPU será lento')


Dispositivo: cuda
  GPU:  NVIDIA GeForce GTX 1650
  VRAM: 4.3 GB


In [3]:
# ── Configuração ──────────────────────────────────────────────────────────────
ROOT       = Path('data')
SIGNAL_DIR = ROOT / 'signals'   # saída do NB1: CHBMIT__chb06__ctx001.npz ...
PRE_EST    = ROOT / 'preictal_estimate.json'

DS_TEST  = 'CHBMIT'
PAT_TEST = 'chb06'

# Sinal
SFREQ   = 256.0   # Hz (NB1 reamostrou para 256)
WIN_S   = 4.0     # segundos por janela (menor que NB3=30s para caber na VRAM)
STEP_S  = 2.0     # passo entre janelas
WIN_T   = int(WIN_S  * SFREQ)   # 1024 amostras
STEP_T  = int(STEP_S * SFREQ)   # 512 amostras

# Nível de canais — R4 = 15 canais
# NB3 usa esta ordem: Fp1,F3,C3,P3,F7,T3,T5,Fz,Cz,Pz,Fp2,F4,C4,P4,F8 (primeiros 15)
N_CH_R4 = 15

# Janelas para seleção (em segundos, mesmas do NB4)
WINDOWS_FIXED = {'W1': 10*60, 'W2': 15*60, 'W3': 20*60, 'W4': 25*60}
PRESEC_FALLBACK_S = 10.2 * 60

# LOSO / treino
N_SEEDS    = 1       # mude para 5 para resultado final
SEEDS      = [42]
BATCH_SIZE = 32      # menor que features pois janelas são maiores
EPOCHS     = 50
LR         = 1e-3
PATIENCE   = 10
WEIGHT_DECAY = 1e-4

print(f'Paciente:  {DS_TEST}/{PAT_TEST}')
print(f'Janela:    {WIN_S}s = {WIN_T} amostras | Passo: {STEP_S}s = {STEP_T} amostras')
print(f'Canais:    {N_CH_R4} (R4)')
print(f'Input DL:  (batch, {N_CH_R4}, {WIN_T})')
mem_mb = BATCH_SIZE * N_CH_R4 * WIN_T * 4 / 1e6
print(f'VRAM/batch: {mem_mb:.1f} MB  (seguro para 4GB)')


Paciente:  CHBMIT/chb06
Janela:    4.0s = 1024 amostras | Passo: 2.0s = 512 amostras
Canais:    15 (R4)
Input DL:  (batch, 15, 1024)
VRAM/batch: 2.0 MB  (seguro para 4GB)


In [5]:
# ── Carregamento dos sinais brutos do NB1 ────────────────────────────────────
# NB1 salva: data/signals/CHBMIT__chb06__ctx001.npz, ctx002.npz, ...
# Cada .npz tem:
#   inter: (n_ch, n_amostras_inter)  — sinal interictal filtrado 0.5-40Hz, 256Hz
#   pre:   (n_ch, n_amostras_pre)    — sinal pré-ictal filtrado
#   ch_names, sfreq, meta

def segment_signal(sig, win_t, step_t, n_ch_max=None):
    """
    sig: (n_ch, n_amostras)
    Retorna: (n_janelas, n_ch, win_t)
    """    
    n_ch, n_t = sig.shape
    if n_ch_max is not None:
        n_ch = min(n_ch, n_ch_max)
        sig  = sig[:n_ch]
    if n_t < win_t:
        return np.zeros((0, n_ch, win_t), dtype=np.float32)
    starts = np.arange(0, n_t - win_t + 1, step_t)
    windows = np.stack([sig[:, s:s+win_t] for s in starts])  # (n_win, n_ch, win_t)
    return windows.astype(np.float32)

def zscore_signal(sig):
    """Z-score por canal (eixo temporal)."""    
    mu = sig.mean(axis=-1, keepdims=True)
    sd = sig.std(axis=-1, keepdims=True) + 1e-10
    return (sig - mu) / sd

def load_context_signal(npz_path, n_ch_max=N_CH_R4, win_t=WIN_T, step_t=STEP_T):
    """Carrega .npz do NB1 e segmenta em janelas."""    
    raw  = np.load(npz_path, allow_pickle=True)
    meta = json.loads(str(raw['meta']))
    ctx_id = meta['contexto_id']

    inter = raw['inter'].astype(np.float32)   # (n_ch, n_t_inter)
    pre   = raw['pre'].astype(np.float32)     # (n_ch, n_t_pre)

    # Z-score por canal antes de segmentar
    inter = zscore_signal(inter)
    pre   = zscore_signal(pre)

    # Segmenta em janelas
    Xi = segment_signal(inter, win_t, step_t, n_ch_max)  # (n_win_i, n_ch, win_t)
    Xp = segment_signal(pre,   win_t, step_t, n_ch_max)  # (n_win_p, n_ch, win_t)

    return Xp, Xi, ctx_id, meta

# Carrega todos os contextos de chb06
npz_files = sorted(SIGNAL_DIR.glob(f'{DS_TEST}__{PAT_TEST}__ctx*.npz'))
if not npz_files:
    raise FileNotFoundError(
        f'Nenhum .npz encontrado em {SIGNAL_DIR} para {DS_TEST}/{PAT_TEST}\n'
        f'Execute NB1 primeiro.'
    )

all_Xp, all_Xi = [], []
all_cp, all_ci = [], []
ctx_map = {}   # ctx_id -> meta

for fp in npz_files:
    Xp, Xi, ctx_id, meta = load_context_signal(fp)
    if len(Xp) == 0 or len(Xi) == 0:
        print(f'  AVISO: {fp.name} — janelas insuficientes (pre={len(Xp)}, inter={len(Xi)})')
        continue
    all_Xp.append(Xp); all_Xi.append(Xi)
    all_cp.append(np.full(len(Xp), ctx_id, dtype=np.int32))
    all_ci.append(np.full(len(Xi), ctx_id, dtype=np.int32))
    ctx_map[ctx_id] = meta
    print(f'  ctx{ctx_id:03d}: pre={Xp.shape} inter={Xi.shape}')

X_pre         = np.vstack(all_Xp)   # (total_win_pre, n_ch, win_t)
X_inter       = np.vstack(all_Xi)
ctx_ids_pre   = np.concatenate(all_cp)
ctx_ids_inter = np.concatenate(all_ci)
contexts      = sorted(set(ctx_ids_pre) & set(ctx_ids_inter))

print()
print(f'{DS_TEST}/{PAT_TEST} carregado:')
print(f'  {len(npz_files)} contextos | {len(contexts)} válidos para LOSO')
print(f'  X_pre:   {X_pre.shape}   (janelas pré-ictal)')
print(f'  X_inter: {X_inter.shape}  (janelas interictal)')
print(f'  Memória: {(X_pre.nbytes + X_inter.nbytes)/1e6:.1f} MB')

# PRE_SEC individual (usado só como referência — janela já é W fixo no raw)
PRE_SEC_PATIENT = None
if PRE_EST.exists():
    for e in json.loads(PRE_EST.read_text()):
        if e['dataset'] == DS_TEST and e['paciente'] == PAT_TEST:
            PRE_SEC_PATIENT = e.get('pre_sec')
            break
print(f'  PRE_SEC individual: {(PRE_SEC_PATIENT or PRESEC_FALLBACK_S)/60:.1f} min')


  ctx001: pre=(899, 15, 1024) inter=(4499, 15, 1024)
  ctx002: pre=(899, 15, 1024) inter=(4499, 15, 1024)
  ctx003: pre=(899, 15, 1024) inter=(4499, 15, 1024)
  ctx004: pre=(899, 15, 1024) inter=(4499, 15, 1024)
  ctx005: pre=(899, 15, 1024) inter=(4499, 15, 1024)
  ctx006: pre=(899, 15, 1024) inter=(4499, 15, 1024)
  ctx007: pre=(899, 15, 1024) inter=(4499, 15, 1024)

CHBMIT/chb06 carregado:
  7 contextos | 7 válidos para LOSO
  X_pre:   (6293, 15, 1024)   (janelas pré-ictal)
  X_inter: (31493, 15, 1024)  (janelas interictal)
  Memória: 2321.6 MB
  PRE_SEC individual: 13.7 min


In [6]:
# ── Filtro de janela para sinal bruto ────────────────────────────────────────
# No NB3/NB4: t_pre é negativo, filtra últimos W segundos.
# Com sinal bruto, o NPZ do NB1 já contém APENAS o trecho relevante
# (pre_inicio_s até onset, inter_inicio_s até inter_fim_s).
# Não há referencial t — usamos as ÚLTIMAS janelas do sinal pré-ictal
# para simular o filtro de W segundos.

def filter_window_raw(X, ctx_ids, W_s, sfreq=SFREQ, win_t=WIN_T, step_t=STEP_T):
    """
    Mantém apenas as últimas W_s segundos de cada contexto.
    X: (n_total_win, n_ch, win_t)
    ctx_ids: (n_total_win,)
    Retorna: X_filtered, ctx_ids_filtered
    """
    n_wins_max = int(W_s / (step_t / sfreq))  # número máx de janelas em W_s
    out_X, out_c = [], []
    for ctx_id in np.unique(ctx_ids):
        mask = ctx_ids == ctx_id
        Xc   = X[mask]
        # pega as últimas n_wins_max janelas
        Xc_filt = Xc[-n_wins_max:] if len(Xc) > n_wins_max else Xc
        out_X.append(Xc_filt)
        out_c.append(np.full(len(Xc_filt), ctx_id, dtype=np.int32))
    return np.vstack(out_X), np.concatenate(out_c)

def undersample_11(Xp, Xi, seed):
    rng = np.random.default_rng(seed)
    n   = min(len(Xp), len(Xi))
    ip  = rng.choice(len(Xp), n, replace=False)
    ii  = rng.choice(len(Xi), n, replace=False)
    return np.vstack([Xp[ip], Xi[ii]]), np.concatenate([np.ones(n), np.zeros(n)])

def compute_metrics(y_true, y_prob, n_inter_test):
    if len(np.unique(y_true)) < 2:
        return dict(auc=np.nan, f1=np.nan, sensitivity=np.nan,
                    specificity=np.nan, fp_h=np.nan)
    auc  = roc_auc_score(y_true, y_prob)
    pred = (y_prob >= 0.5).astype(int)
    f1   = f1_score(y_true, pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    sens = tp/(tp+fn) if (tp+fn)>0 else 0.
    spec = tn/(tn+fp) if (tn+fp)>0 else 0.
    dur_h = n_inter_test * STEP_S / 3600.
    fph   = fp/dur_h if dur_h>0 else np.nan
    return dict(auc=auc, f1=f1, sensitivity=sens, specificity=spec, fp_h=fph)

print('Utilitários definidos.')
print(f'  Janela máxima W4=25min → {int(25*60/(STEP_S))} janelas de {WIN_S}s')


Utilitários definidos.
  Janela máxima W4=25min → 750 janelas de 4.0s


In [7]:
# ── Arquiteturas para sinal bruto ────────────────────────────────────────────

class EEGNet(nn.Module):
    """
    EEGNet (Lawhern et al. 2018) — arquitetura específica para EEG.
    ~2.5K parâmetros, projetada para datasets pequenos.
    Entrada: (batch, 1, n_ch, T)
    """
    def __init__(self, n_ch=N_CH_R4, T=WIN_T,
                 F1=8, D=2, F2=16, dropout=0.5):
        super().__init__()
        self.temporal = nn.Sequential(
            # Convolução temporal: aprende filtros espectrais
            nn.Conv2d(1, F1, (1, T//4), padding=(0, T//8), bias=False),
            nn.BatchNorm2d(F1),
        )
        self.depthwise = nn.Sequential(
            # Convolução espacial: combina canais (DepthwiseConv)
            nn.Conv2d(F1, F1*D, (n_ch, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1*D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )
        self.separable = nn.Sequential(
            # Convolução separável: captura dependências temporais
            nn.Conv2d(F1*D, F2, (1, T//32), padding=(0, T//64), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout),
        )
        # Calcula tamanho do flatten automaticamente
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_ch, T)
            x = self.temporal(dummy)
            x = self.depthwise(x)
            x = self.separable(x)
            flat_size = x.view(1, -1).shape[1]
        self.head = nn.Linear(flat_size, 1)

    def forward(self, x):
        # x: (batch, n_ch, T) → (batch, 1, n_ch, T)
        x = x.unsqueeze(1)
        x = self.temporal(x)
        x = self.depthwise(x)
        x = self.separable(x)
        return self.head(x.view(x.shape[0], -1)).squeeze(-1)


class CNN1DRaw(nn.Module):
    """
    CNN-1D temporal sobre sinal EEG bruto.
    Entrada: (batch, n_ch, T)
    Aprende filtros no eixo do tempo (equivale a filtros de banda).
    """
    def __init__(self, n_ch=N_CH_R4, T=WIN_T, dropout=0.3):
        super().__init__()
        self.conv = nn.Sequential(
            # Filtros longos: capturam padrões de baixa frequência (delta, theta)
            nn.Conv1d(n_ch, 32, kernel_size=64, stride=4, padding=32),
            nn.BatchNorm1d(32), nn.ELU(),
            nn.Conv1d(32, 64, kernel_size=16, stride=2, padding=8),
            nn.BatchNorm1d(64), nn.ELU(),
            nn.Conv1d(64, 128, kernel_size=8, stride=2, padding=4),
            nn.BatchNorm1d(128), nn.ELU(),
            nn.AdaptiveAvgPool1d(8),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*8, 64), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.head(self.conv(x)).squeeze(-1)


DL_MODELS = ['EEGNet', 'CNN1D_Raw']

def make_model(name):
    if name == 'EEGNet':   return EEGNet(N_CH_R4, WIN_T).to(DEVICE)
    if name == 'CNN1D_Raw': return CNN1DRaw(N_CH_R4, WIN_T).to(DEVICE)
    raise ValueError(name)

print('Arquiteturas:')
for name in DL_MODELS:
    m = make_model(name)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'  {name:12s}: {n:,} parâmetros')


Arquiteturas:
  EEGNet      : 11,073 parâmetros
  CNN1D_Raw   : 195,361 parâmetros


In [8]:
# ── Função de treino ─────────────────────────────────────────────────────────

def train_eval_dl(model_name, Xtr, ytr, Xte):
    """
    Xtr, Xte: (n, n_ch, win_t) — sinal bruto já normalizado
    ytr: (n,) float32
    Retorna: probabilidades no teste
    """
    model  = make_model(model_name)
    ds_tr  = TensorDataset(torch.tensor(Xtr, dtype=torch.float32),
                           torch.tensor(ytr, dtype=torch.float32))
    loader = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True)
    Xte_t  = torch.tensor(Xte, dtype=torch.float32).to(DEVICE)

    n_pos = ytr.sum(); n_neg = len(ytr) - n_pos
    pos_w = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(DEVICE)
    crit  = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt   = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    best_loss, patience_cnt, best_state = float('inf'), 0, None
    model.train()
    for ep in range(EPOCHS):
        ep_loss = 0.
        for Xb, yb in loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(Xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_loss += loss.item() * len(Xb)
        ep_loss /= len(ds_tr)
        sched.step()
        if ep_loss < best_loss - 1e-4:
            best_loss = ep_loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE: break

    if best_state: model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(Xte_t)).cpu().numpy()
    return probs

print('train_eval_dl definido.')


train_eval_dl definido.


In [9]:
# ── LOSO — sinal bruto ───────────────────────────────────────────────────────
# Seleção de janela W por inner LOSO (determina quantas janelas usar de cada ctx)
# Outer fold: treina EEGNet e CNN1D_Raw
# Opção B: inter completo do contexto de teste no FP/h

results = {m: [] for m in DL_MODELS}
t0 = time.time()

for test_ctx in tqdm(contexts, desc='LOSO', unit='fold'):
    train_ctxs = [c for c in contexts if c != test_ctx]
    if len(train_ctxs) < 1: continue

    # ── Seleção de janela: testa W1..W5 por AUC no inner loop ────────────────
    # Usa a métrica de n_janelas como proxy (mais simples para sinal bruto)
    # A janela W define quantas janelas de cada contexto são usadas
    WINDOWS = dict(WINDOWS_FIXED)
    WINDOWS['W5'] = PRE_SEC_PATIENT if PRE_SEC_PATIENT else PRESEC_FALLBACK_S

    best_wlabel, best_W_s = 'W5', WINDOWS.get('W5', WINDOWS['W1'])

    # ── Monta treino ─────────────────────────────────────────────────────────
    Xp_tr_filt, cp_tr = filter_window_raw(X_pre,   ctx_ids_pre,   best_W_s)
    Xi_tr_filt, ci_tr = filter_window_raw(X_inter, ctx_ids_inter, best_W_s)
    Xp_tr = Xp_tr_filt[np.isin(cp_tr, train_ctxs)]
    Xi_tr = Xi_tr_filt[np.isin(ci_tr, train_ctxs)]

    # ── Monta teste (Opção B: inter completo) ────────────────────────────────
    Xp_te = X_pre[ctx_ids_pre == test_ctx]
    Xi_te = X_inter[ctx_ids_inter == test_ctx]   # inter completo, sem filtro W

    if any(len(a) == 0 for a in [Xp_tr, Xi_tr, Xp_te, Xi_te]): continue

    X_test = np.vstack([Xp_te, Xi_te])
    y_test = np.concatenate([np.ones(len(Xp_te)), np.zeros(len(Xi_te))])
    n_inter_test = len(Xi_te)

    for seed in SEEDS:
        torch.manual_seed(seed); np.random.seed(seed)
        Xtr, ytr = undersample_11(Xp_tr, Xi_tr, seed)

        # Normalização por canal (z-score sobre o eixo temporal da janela)
        # Xtr shape: (n_win, n_ch, win_t)
        mu = Xtr.mean(axis=(0,2), keepdims=True)
        sd = Xtr.std(axis=(0,2), keepdims=True) + 1e-10
        Xtr_s = ((Xtr - mu) / sd).astype(np.float32)
        Xte_s = ((X_test - mu) / sd).astype(np.float32)

        for model_name in DL_MODELS:
            try:
                probs = train_eval_dl(model_name, Xtr_s, ytr, Xte_s)
                if len(np.unique(y_test)) > 1:
                    m = compute_metrics(y_test, probs, n_inter_test)
                    results[model_name].append({
                        'fold_ctx': test_ctx,
                        'chosen_window': best_wlabel,
                        'chosen_W_s': best_W_s, **m
                    })
            except Exception as e:
                print(f'  {model_name} fold={test_ctx} seed={seed}: {e}')

elapsed = time.time() - t0
print(f'\nLOSO concluído em {elapsed/60:.1f} min')
for name in DL_MODELS:
    df = pd.DataFrame(results[name])
    if not df.empty:
        print(f'{name:12s}: AUC={df["auc"].mean():.4f}±{df["auc"].std():.4f} '
              f'Sens={df["sensitivity"].mean():.4f} '
              f'Spec={df["specificity"].mean():.4f} '
              f'FP/h={df["fp_h"].mean():.1f}')


LOSO:   0%|          | 0/7 [00:00<?, ?fold/s]

  EEGNet fold=1 seed=42: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.



KeyboardInterrupt: 

In [ ]:
# ── Tabela de resultados ──────────────────────────────────────────────────────
rows = []
for name in DL_MODELS:
    df = pd.DataFrame(results[name])
    if df.empty: continue
    agg = df.groupby('fold_ctx').mean(numeric_only=True).reset_index()
    rows.append({
        'Modelo': name,
        'AUC':    round(agg['auc'].mean(), 4),
        'F1':     round(agg['f1'].mean(), 4),
        'Sens':   round(agg['sensitivity'].mean(), 4),
        'Spec':   round(agg['specificity'].mean(), 4),
        'FP/h':   round(agg['fp_h'].mean(), 1),
        'Folds':  len(agg),
    })

df_res = pd.DataFrame(rows).sort_values('AUC', ascending=False).reset_index(drop=True)
pd.set_option('display.width', 100)
print('=' * 70)
print(f'RESULTADOS — {DS_TEST}/{PAT_TEST} | sinal bruto | {len(contexts)} folds | {N_SEEDS} seed(s)')
print('=' * 70)
print(df_res.to_string(index=False))
print()
print('Notas:')
print(f'  Janela EEG: {WIN_S}s / {WIN_T} amostras @ {SFREQ}Hz')
print(f'  Canais: {N_CH_R4} (R4) | Passo: {STEP_S}s')
print( '  FP/h: inter completo do teste (Opção B)')
print( '  EEGNet: ~2.5K params, projetado para EEG com n pequeno (Lawhern 2018)')
print( '  CNN1D_Raw: filtros temporais longos (64 amostras = 0.25s)')
